<a href="https://colab.research.google.com/github/phenxie/1/blob/main/ollamaDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://blog.csdn.net/weixin_66401877/article/details/145892615

这些库包括 Unsloth 主程序、bitsandbytes（用于量化模型）和 unsloth_zoo（预训练模型支持）。

In [2]:
%%capture
# 这是一个 Jupyter Notebook 的魔法命令，用于隐藏命令的输出。
# 通过捕获输出，可以让 Colab 的界面更整洁，避免显示冗长的安装日志。

# 安装 unsloth 包。
# unsloth 是一个高效的工具，用于微调大型语言模型（LLM），能显著减少显存需求并加速训练。
!pip install unsloth

# 卸载当前已安装的 unsloth 包（如果存在），然后从 GitHub 安装最新版本。
# "-y" 表示自动确认卸载，"--upgrade" 确保获取最新版，"--no-cache-dir" 避免使用缓存，
# "--no-deps" 跳过依赖安装（因为我们只关心 unsloth 本身），
# 通过 GitHub 源安装可以获得最新的功能和修复。
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# 安装 bitsandbytes 和 unsloth_zoo 两个依赖包。
# bitsandbytes 是一个用于模型量化的库，支持 4 位和 8 位精度，能大幅降低内存占用。
# unsloth_zoo 提供了一些预训练模型或相关工具，方便用户快速上手。
!pip install bitsandbytes unsloth_zoo


 加载预训练模型
选择一个基础模型，这里用 unsloth/DeepSeek-R1-Distill-Llama-8B：

In [3]:
# 导入 Unsloth 提供的 FastLanguageModel 类，用于加载和操作高效的大型语言模型。
from unsloth import FastLanguageModel

# 导入 PyTorch 库，它是深度学习的基础框架，用于处理模型的张量计算和 GPU 加速。
import torch

# 设置模型处理文本的最大序列长度（单位：token），这里设为 2048。
# 这决定了模型一次能处理的文本长度，越大越能捕捉长上下文，但也增加显存需求。
max_seq_length = 2048

# 设置模型的数据类型（dtype），这里设为 None 表示让 Unsloth 自动选择最优类型。
# 通常会根据硬件支持选择 float16 或 bfloat16，以平衡精度和性能。
dtype = None

# 启用 4 位量化加载模型，值为 True。
# 4 位量化可以将模型大小和显存需求减少约 75%，非常适合在资源有限的环境（如 Colab 免费版）运行。
load_in_4bit = True

# 从预训练模型库中加载指定的模型和对应的 tokenizer（分词器）。
# 返回两个对象：model（模型本身）和 tokenizer（用于将文本转为数字输入的工具）。
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",  # 指定模型名称，这里使用 Unsloth 优化的 DeepSeek-R1-Distill-Llama-8B。
    max_seq_length=max_seq_length,                      # 使用上面定义的最大序列长度。
    dtype=dtype,                                       # 使用上面定义的数据类型（自动选择）。
    load_in_4bit=load_in_4bit,                         # 启用 4 位量化加载。
    # token="hf_...",                                  # 如果需要访问私有模型，可以取消注释并填入 Hugging Face 的 API 令牌。
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.2: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

微调前测试
先看看模型未经训练的表现：
这里我们调用推理模型进行推理，输出可能是泛泛而谈，微调后会更精准。

In [4]:
# 定义提示模板（prompt_style），这是一个多行字符串，用于格式化输入和输出。
# 模板包含指令、问题和回答部分，设计目的是引导模型生成结构化的回答。
prompt_style = """以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通医学知识的医生，能够回答关于疾病、治疗方案和健康建议的问题。
请回答以下医疗问题。

### 问题：
{}

### 回答：
<think>{}"""

# 定义一个测试问题，用于在微调前检查模型的初始能力。
# 这里选择了一个常见的医疗问题，方便观察模型的表现。
question = "我最近总是感到疲劳，可能是什么原因？"

# 将模型切换到推理模式。
# FastLanguageModel.for_inference 是 Unsloth 提供的方法，优化模型以进行生成任务，避免训练时的额外开销。
FastLanguageModel.for_inference(model)

# 使用 tokenizer 将格式化的提示转换为模型可处理的数字输入。
# prompt_style.format(question, "") 将问题插入模板，思考部分暂时为空（留给模型生成）。
# return_tensors="pt" 表示返回 PyTorch 张量格式，to("cuda") 将数据移到 GPU 上加速处理。
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 调用模型生成回答。
# input_ids 是编码后的输入序列，attention_mask 指示哪些部分需要关注，
# max_new_tokens=1200 限制生成最多 1200 个新 token，use_cache=True 启用缓存以加速生成。
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

# 将模型生成的数字输出解码为人类可读的文本。
# batch_decode 处理批量输出，这里取第一个（也是唯一一个）结果。
response = tokenizer.batch_decode(outputs)

# 打印生成的回答，展示模型在微调前的能力。
# response[0] 是解码后的完整文本，可能包含提示部分和生成的回答。
print(response[0])


<｜begin▁of▁sentence｜>以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通医学知识的医生，能够回答关于疾病、治疗方案和健康建议的问题。
请回答以下医疗问题。

### 问题：
我最近总是感到疲劳，可能是什么原因？

### 回答：
<think>
最近总是感到疲劳，可能是由于多种原因引起的。首先，我要考虑的是生活方式。最近是不是工作压力大，或者睡眠不足？如果是这样，可能需要调整作息，保证充足的睡眠时间。其次，饮食方面也很重要。最近有没有改变饮食习惯，摄入更多高热量或缺乏维生素和矿物质的食物？如果有，可能需要调整饮食，补充必要的营养。再者，身体的运动量是否增加？如果最近开始频繁锻炼或做了大量的体力活，可能会导致疲劳。此外，压力和焦虑也是导致疲劳的重要因素。如果有长期的压力或情绪问题，可能需要寻求专业的心理咨询。另外，某些慢性疾病，比如贫血、甲状腺功能异常、糖尿病等，也可能导致持续的疲劳。如果有这些情况，建议尽早就医进行检查。最后，药物的影响也是一个方面。如果最近开始服用新药物或增加药物剂量，可能会产生疲劳作为副作用。总之，建议先进行全面的体检，了解血液检查结果，排除疾病的可能性。如果没有明显的身体疾病，可能需要调整生活方式和作息，适当减少压力，增加运动和休息，来改善疲劳感。
</think>

您最近感到疲劳，可能的原因很多，以下是一些可能的原因和建议：

1. **睡眠不足或质量差**：确保每天有足够的睡眠时间，尤其是充足的 REM 却眠。如果您最近睡眠时间减少或睡眠质量不好，可能导致疲劳。

2. **饮食不均衡**：多摄入高热量食物或缺乏维生素和矿物质的食物可能会导致能量不足，进而感到疲劳。建议多吃蔬菜、水果和全谷物，保持饮食的均衡。

3. **运动量增加**：如果最近开始频繁锻炼或做了大量的体力活动，可能会导致身体疲劳。确保运动量适中，避免过度训练。

4. **压力和焦虑**：长期的压力和焦虑会消耗大量的能量，导致疲劳。尝试通过冥想、深呼吸、瑜伽等方式来缓解压力。

5. **贫血或缺乏维生素**：贫血或缺乏铁、维生素B12等维生素会导致疲劳。建议定期吃富含铁和维生素的食物，或者在医生指导下服用补

加载与格式化数据集
使用 shibing624/medical 中文医疗数据集：
我在这里定义了一个叫做 train_prompt_style 的字符串模板，它就像是我给模型的一份“任务说明书”。我的目的是告诉模型，它现在是一位精通医学知识的医生，需要回答医疗相关的问题。这个模板分为几个部分：首先，我写了一个总体的任务描述，提醒模型要认真思考并给出准确的回答；接着，我明确了指令，设定模型的角色和任务；然后，我用 {} 留了三个占位符，分别用来填入具体的问题、思考过程和最终的回答。这样，我就能确保模型在训练时按照这个结构生成内容，比如先分析问题，再给出专业建议。这样设计既清晰又有逻辑，方便模型学习如何像医生一样思考和回应。


In [6]:
train_prompt_style = """以下是描述任务的指令，以及提供进一步上下文的输入。
请写出一个适当完成请求的回答。
在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。

### 指令：
你是一位精通医学知识的医生，能够回答关于疾病、治疗方案和健康建议的问题。
请回答以下医疗问题。

### 问题：
{}

### 回答：
<思考>
{}
</思考>
{}"""


我在这段代码里主要是为了准备训练数据，让模型能理解和生成医疗相关的回答。首先，我定义了一个结束标记EOS_TOKEN，这是从分词器里拿来的，用来告诉模型每段文本到哪里结束。接着，我从 datasets 库里导入了 load_dataset 函数，用它加载了一个医疗数据集 shibing624/medical，选了finetune 配置里的前 200 条训练数据。为了搞清楚数据长什么样，我打印了它的字段名，结果是 instruction、input 和 output。然后，我写了一个函数 formatting_prompts_func，用来把这些数据格式化成我想要的样子：我从数据里抽出instruction 作为问题，input 作为思考过程，output 作为回答，再用之前定义的 train_prompt_style 模板把它们组合起来，加上结束标记，最后存进一个列表里。通过 dataset.map 方法，我批量处理了所有数据，最后输出了第一条格式化后的文本，看看是不是按我的预期工作。这一步的目的是让数据变成模型能直接学习的格式，确保训练顺利进行。


In [7]:
# 定义结束标记（EOS_TOKEN），用于指示文本的结束
EOS_TOKEN = tokenizer.eos_token  # 必须添加结束标记

# 导入数据集加载函数
from datasets import load_dataset
# 加载指定的数据集，选择中文语言和训练集的前500条记录
dataset = load_dataset("shibing624/medical", 'finetune', split = "train[0:200]", trust_remote_code=True)
# 打印数据集的列名，查看数据集中有哪些字段
print(dataset.column_names)


README.md:   0%|          | 0.00/9.14k [00:00<?, ?B/s]

medical.py:   0%|          | 0.00/7.11k [00:00<?, ?B/s]

train_zh_0.json:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

train_en_1.json:   0%|          | 0.00/139M [00:00<?, ?B/s]

valid_zh_0.json:   0%|          | 0.00/307k [00:00<?, ?B/s]

...

test_en_1.json:   0%|          | 0.00/602k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

['instruction', 'input', 'output']


In [8]:
# 定义一个函数，用于格式化数据集中的每条记录
def formatting_prompts_func(examples):
    # 从数据集中提取问题、复杂思考过程和回答
    inputs = examples["instruction"]
    cots = examples["input"]
    outputs = examples["output"]
    texts = []  # 用于存储格式化后的文本
    # 遍历每个问题、思考过程和回答，进行格式化
    for input, cot, output in zip(inputs, cots, outputs):
        # 使用字符串模板插入数据，并加上结束标记
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)  # 将格式化后的文本添加到列表中
    return {
        "text": texts,  # 返回包含所有格式化文本的字典
    }

dataset = dataset.map(formatting_prompts_func, batched = True)
dataset["text"][0]


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

'以下是描述任务的指令，以及提供进一步上下文的输入。\n请写出一个适当完成请求的回答。\n在回答之前，请仔细思考问题，并创建一个逻辑连贯的思考过程，以确保回答准确无误。\n\n### 指令：\n你是一位精通医学知识的医生，能够回答关于疾病、治疗方案和健康建议的问题。\n请回答以下医疗问题。\n\n### 问题：\n血热的临床表现是什么?\n\n### 回答：\n<思考>\n\n</思考>\n初发或复发病不久。皮疹发展迅速，呈点滴状、钱币状或混合状。常见丘疹、斑丘疹、大小不等的斑片，潮红、鲜红或深红色。散布于体表各处或几处，以躯干、四肢多见，亦可先从头面开始，逐渐发展至全身。新皮疹不断出现，表面覆有银白色鳞屑，干燥易脱落，剥刮后有点状出血。可有同形反应;伴瘙痒、心烦口渴。大便秘结、小便短黄，舌质红赤，苔薄黄或根部黄厚，脉弦滑或滑数。血热炽盛病机，主要表现在如下四个面：一、热象：血热多属阳盛则热之实性、热性病机和病证、并表现出热象。二、血行加速：血得热则行，可使血流加速，且使脉道扩张，络脉充血，故可见面红目赤，舌色深红（即舌绛）等症。三、动血：在血行加速与脉道扩张的基础上，血分有热，可灼伤脉络，引起出血，称为“热迫血妄行”，或称动血。四、扰乱心神：血热炽盛则扰动心神，心主血脉而藏神，血脉与心相通，故血热则使心神不安，而见心烦，或躁扰发狂等症。<｜end▁of▁sentence｜>'

执行微调训练
配置 LoRA 并开始训练：
训练约需 20-30 分钟，视数据量而定。

In [12]:
# 将模型切换到训练模式。
# FastLanguageModel.for_training 是 Unsloth 提供的方法，确保模型准备好进行参数更新，而不是仅用于推理。
FastLanguageModel.for_training(model)

# 配置并返回一个支持参数高效微调（PEFT）的模型。
# get_peft_model 使用 LoRA 技术，只更新模型的部分参数，从而减少显存需求和计算开销。
model = FastLanguageModel.get_peft_model(
    model,  # 传入之前加载的预训练模型，作为微调的基础。
    r=16,   # 设置 LoRA 的秩（rank），控制新增可训练参数的规模。值越大，模型调整能力越强，但显存需求也增加。
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            # 指定需要应用 LoRA 的模型模块，这些是 Transformer 架构中的关键部分（如注意力机制和前馈网络）。
    lora_alpha=16,
            # LoRA 的缩放因子，影响新增参数对模型的贡献程度。通常与 r 成比例设置，这里为 16。
    lora_dropout=0,
            # 设置 LoRA 层的 dropout 比率，用于防止过拟合。这里设为 0，表示不丢弃任何参数。
    bias="none",
            # 指定是否为 LoRA 参数添加偏置项。"none" 表示不添加，保持轻量化。
    use_gradient_checkpointing="unsloth",
            # 启用梯度检查点技术，Unsloth 优化版本能节省显存，支持更大的批量大小或模型。
    random_state=3407,
            # 设置随机种子，确保每次运行时模型初始化的随机性一致，便于结果复现。
    use_rslora=False,
            # 是否使用 Rank-Stabilized LoRA（一种改进的 LoRA 变体）。这里设为 False，使用标准 LoRA。
    loftq_config=None,
            # 设置是否使用 LoftQ（一种量化技术）。这里设为 None，表示不启用。
)


Unsloth: Already have LoRA adapters! We shall skip this step.


In [14]:
# 导入 SFTTrainer 类，用于监督微调训练。
# SFTTrainer 是 TRL（Transformers Reinforcement Learning）库提供的工具，适合基于指令的数据集微调模型。
from trl import SFTTrainer

# 导入 TrainingArguments 类，用于定义训练的超参数。
# 这是 Hugging Face Transformers 库的核心类，允许灵活配置训练过程。
from transformers import TrainingArguments

# 导入 Unsloth 提供的函数，用于检查硬件是否支持 bfloat16 数据类型。
# bfloat16 是一种高效的半精度格式，能在支持的硬件上加速训练。
from unsloth import is_bfloat16_supported

# 创建 SFTTrainer 实例，配置训练所需的模型、数据和参数。
trainer = SFTTrainer(
    model=model,                    # 传入之前配置好的模型（已启用 LoRA）。
    tokenizer=tokenizer,            # 传入与模型配套的分词器，用于处理文本数据。
    train_dataset=dataset,          # 传入训练数据集（已格式化为包含 "text" 字段）。
    dataset_text_field="text",      # 指定数据集中包含训练文本的字段名，这里是 "text"。
    max_seq_length=max_seq_length,  # 设置最大序列长度，与模型加载时保持一致（如 2048）。
    dataset_num_proc=2,             # 设置数据处理的并行进程数，加速数据预处理。
    packing=False,                  # 是否启用序列打包。False 表示不打包，每条数据独立处理。
    args=TrainingArguments(         # 定义训练超参数的配置对象。
        per_device_train_batch_size=2,  # 每个设备（GPU）的批次大小，设为 2 以适应显存限制。
        gradient_accumulation_steps=4,  # 梯度累积步数，累积 4 次小批次后更新参数，模拟大批量训练。
        warmup_steps=5,                 # 预热步数，学习率在前 5 步逐渐增加，稳定训练。
        max_steps=75,                   # 最大训练步数，控制训练时长（步数 = 数据量 / 批次大小）。
        learning_rate=2e-4,             # 学习率，设为 0.0002，控制参数更新速度。
        fp16=not is_bfloat16_supported(),  # 如果不支持 bfloat16，则使用 fp16（16 位浮点数）加速训练。
        bf16=is_bfloat16_supported(),      # 如果硬件支持 bfloat16，则启用它，兼顾精度和速度。
        logging_steps=1,                   # 每 1 步记录一次训练日志，方便监控损失变化。
        optim="adamw_8bit",                # 使用 8 位 AdamW 优化器，节省显存并保持性能。
        weight_decay=0.01,                 # 权重衰减系数，设为 0.01，防止模型过拟合。
        lr_scheduler_type="linear",        # 学习率调度器类型，"linear" 表示线性衰减。
        seed=3407,                         # 随机种子，确保训练结果可复现。
        output_dir="outputs",              # 训练结果（如检查点）保存的目录。
        report_to="none",                  # 不将训练日志报告到外部工具（如 WandB），仅本地记录。
    ),
)

# 开始训练模型。
# trainer.train() 执行微调过程，根据配置更新模型参数，完成后保存到 output_dir。
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss


RuntimeError: PassManager::run failed

微调后测试
用相同问题测试效果：
回答更贴近医疗专业知识。

In [ ]:
print(question) # 打印前面的问题
# 将模型切换到推理模式，准备回答问题
FastLanguageModel.for_inference(model)

# 将问题转换成模型能理解的格式，并发送到 GPU 上
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 让模型根据问题生成回答，最多生成 4000 个新词
outputs = model.generate(
    input_ids=inputs.input_ids,  # 输入的数字序列
    attention_mask=inputs.attention_mask,  # 注意力遮罩，帮助模型理解哪些部分重要
    max_new_tokens=4000,  # 最多生成 4000 个新词
    use_cache=True,  # 使用缓存加速生成
)

# 将生成的回答从数字转换回文字
response = tokenizer.batch_decode(outputs)

# 打印回答
print(response[0])


将微调后的模型保存为 GGUF 格式
GGUF，全称是 GPT-Generated Unified Format（GPT 生成的统一格式），是一种专门为存储和部署大型语言模型（LLM）设计的文件格式。简单来说，它就像一个“打包盒”，把模型的所有必要信息都装在一起，比如模型的权重（参数）、分词器（tokenizer）信息、超参数（hyperparameters）和元数据（metadata），全都压缩成一个二进制文件。这样做的好处是方便高效地加载和运行模型，尤其是在本地设备上。
执行前我们需要先在Colab中配置token的环境变量

In [ ]:
# 导入 Google Colab 的 userdata 模块，用于访问用户数据
from google.colab import userdata

# 从 Google Colab 用户数据中获取 Hugging Face 的 API 令牌
HUGGINGFACE_TOKEN = userdata.get('HUGGINGFACE_TOKEN')

# 将模型保存为 8 位量化格式（Q8_0）
# 这种格式文件小且运行快，适合部署到资源受限的设备
if True: model.save_pretrained_gguf("model", tokenizer,)

# 将模型保存为 16 位量化格式（f16）
# 16 位量化精度更高，但文件稍大
if False: model.save_pretrained_gguf("model_f16", tokenizer, quantization_method = "f16")

# 将模型保存为 4 位量化格式（q4_k_m）
# 4 位量化文件最小，但精度可能稍低
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")


配置HUGGINGFACE_TOKEN的环境变量
Huggingface官网：huggingface
首先我们先获取到Huggingface的token